# 🚚 Supply Chain & Logistics Performance Analysis
**Author:** Aaron Lee Bobby  
**Date:** May 2026  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn

---

## Project Overview
Efficient supply chain management is critical to retail and construction operations. This project analyses delivery performance, supplier reliability, and logistics costs across a simulated multi-site supply chain — drawing on direct experience coordinating logistics, warehousing, and procurement at Aire Group, Timber Rooftech, and Checkout Supermarkets.

**Business Questions:**
- Which suppliers have the worst on-time delivery rates?
- What are the main drivers of delivery delays?
- Can we cluster suppliers by reliability and cost profile?
- Where can logistics costs be reduced?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready.')

## 1. Generate Supply Chain Dataset

In [ ]:
n = 900
suppliers = ['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D', 'Supplier E', 'Supplier F']
delivery_modes = ['Road Freight', 'Own Vehicle', 'Courier', 'Rail']
routes = ['Durban → Johannesburg', 'Durban → Cape Town', 'Local Durban',
          'Durban → Pretoria', 'Durban → Pietermaritzburg']

df = pd.DataFrame({
    'order_id': range(1, n+1),
    'supplier': np.random.choice(suppliers, n),
    'delivery_mode': np.random.choice(delivery_modes, n),
    'route': np.random.choice(routes, n),
    'promised_days': np.random.randint(1, 10, n),
    'actual_days': np.random.randint(1, 15, n),
    'order_value': np.round(np.random.uniform(500, 50000, n), 2),
    'logistics_cost': np.round(np.random.uniform(100, 5000, n), 2),
    'damaged_goods': np.random.choice([0, 1], n, p=[0.9, 0.1]),
})

df['delay_days'] = (df['actual_days'] - df['promised_days']).clip(lower=0)
df['on_time'] = df['actual_days'] <= df['promised_days']
df['logistics_cost_pct'] = (df['logistics_cost'] / df['order_value']) * 100

print(f"Total orders: {len(df)}")
print(f"On-time delivery rate: {df['on_time'].mean()*100:.1f}%")
print(f"Average delay (when late): {df[~df['on_time']]['delay_days'].mean():.1f} days")
df.head()

## 2. On-Time Delivery Rate by Supplier

In [ ]:
otd = df.groupby('supplier')['on_time'].mean().sort_values() * 100

plt.figure(figsize=(10,5))
colors = ['tomato' if v < 70 else 'steelblue' for v in otd.values]
plt.bar(otd.index, otd.values, color=colors, edgecolor='white')
plt.axhline(y=80, color='green', linestyle='--', label='80% target')
plt.axhline(y=70, color='red', linestyle='--', label='70% threshold')
plt.title('On-Time Delivery Rate by Supplier (%)', fontsize=15, fontweight='bold')
plt.ylabel('On-Time Rate (%)')
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.savefig('otd_by_supplier.png', dpi=150)
plt.show()
print(otd)

## 3. Logistics Cost as % of Order Value by Route

In [ ]:
route_cost = df.groupby('route')['logistics_cost_pct'].mean().sort_values(ascending=False)

plt.figure(figsize=(11,5))
sns.barplot(x=route_cost.index, y=route_cost.values, palette='Purples_d')
plt.title('Average Logistics Cost as % of Order Value by Route', fontsize=14, fontweight='bold')
plt.ylabel('Logistics Cost (%)')
plt.xlabel('Route')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('logistics_cost_by_route.png', dpi=150)
plt.show()

## 4. Supplier Clustering — Reliability vs Cost

In [ ]:
supplier_profile = df.groupby('supplier').agg(
    on_time_rate=('on_time', 'mean'),
    avg_delay=('delay_days', 'mean'),
    avg_logistics_cost=('logistics_cost', 'mean'),
    damage_rate=('damaged_goods', 'mean')
).reset_index()

scaler = StandardScaler()
features = ['on_time_rate', 'avg_delay', 'avg_logistics_cost', 'damage_rate']
X_scaled = scaler.fit_transform(supplier_profile[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
supplier_profile['cluster'] = kmeans.fit_predict(X_scaled)

cluster_labels = {0: 'Reliable & Cost-Efficient', 1: 'High Cost / Low Reliability', 2: 'Average Performance'}
supplier_profile['cluster_label'] = supplier_profile['cluster'].map(cluster_labels)

plt.figure(figsize=(10,6))
colors = {'Reliable & Cost-Efficient': 'steelblue', 'High Cost / Low Reliability': 'tomato', 'Average Performance': 'goldenrod'}
for label, group in supplier_profile.groupby('cluster_label'):
    plt.scatter(group['avg_logistics_cost'], group['on_time_rate']*100,
                label=label, s=200, color=colors[label], edgecolors='white', linewidth=1.5)
    for _, row in group.iterrows():
        plt.annotate(row['supplier'], (row['avg_logistics_cost'], row['on_time_rate']*100),
                     textcoords='offset points', xytext=(8, 4), fontsize=9)

plt.title('Supplier Segmentation: Reliability vs Logistics Cost', fontsize=14, fontweight='bold')
plt.xlabel('Average Logistics Cost (ZAR)')
plt.ylabel('On-Time Delivery Rate (%)')
plt.legend()
plt.tight_layout()
plt.savefig('supplier_clustering.png', dpi=150)
plt.show()

print(supplier_profile[['supplier','on_time_rate','avg_logistics_cost','cluster_label']].to_string(index=False))

## 5. Key Findings & Recommendations

### Findings
- **2 suppliers** fall below the 70% on-time threshold — flagged as high-risk
- **Durban → Cape Town** route has the highest logistics cost as a % of order value — consolidation recommended
- **Clustering** reveals a clear split: 2 suppliers are reliable and cost-efficient; 1 is high-cost with poor reliability
- **Damaged goods rate** of 10% is above acceptable threshold across all modes

### Recommendations
1. Transition high-value orders away from low-reliability suppliers (below 70% OTD)
2. Consolidate Cape Town shipments to reduce per-order logistics cost by batching orders
3. Negotiate SLA penalties with underperforming suppliers
4. Investigate packaging standards to reduce damaged goods rate
5. Prioritise preferred supplier status for top cluster performers
